# 📰 Fake News Detector (NLP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/fake-news-detector/blob/main/Fake_News_Detector.ipynb)

**Goal:** Kisi bhi news article ko *Real* ya *Fake* classify karna.

**Pipeline:** Data load → EDA → Cleaning → TF-IDF → 4 models → Evaluation → Error analysis → Gradio demo

**Labels:** `0 = Real`, `1 = Fake`

> Notebook ko upar se neeche cells ko order mein run karein (Runtime → Run all).

## 1. Setup

In [ ]:
!pip install -q kagglehub gradio wordcloud

import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                             classification_report)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42
import os
os.makedirs("images", exist_ok=True)
print("Setup done ✅")

## 2. Dataset load karna
Dataset: **Fake and Real News Dataset** (Kaggle). `kagglehub` se seedha download hoga.

Agar download na chale to **Option B** use karein: Kaggle se `archive.zip` download karke Colab mein upload karein.

In [ ]:
# Option A: kagglehub se download
try:
    import kagglehub
    path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
    fake = pd.read_csv(f"{path}/Fake.csv")
    true = pd.read_csv(f"{path}/True.csv")
    print("Dataset kagglehub se load ho gaya ✅")
except Exception as e:
    print("kagglehub fail hua:", e)
    print("Option B chalayein (neeche wala cell).")

In [ ]:
# Option B (sirf zaroorat ho to): zip upload karein
# from google.colab import files
# up = files.upload()            # archive.zip select karein
# !unzip -o archive.zip
# fake = pd.read_csv("Fake.csv"); true = pd.read_csv("True.csv")

In [ ]:
fake["label"] = 1
true["label"] = 0
df = pd.concat([fake, true], ignore_index=True)
print("Shape:", df.shape)
df.head()

## 3. EDA (Exploratory Data Analysis)

In [ ]:
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated(subset=["title", "text"]).sum())

# Duplicates aur khali text hata dein
df = df.drop_duplicates(subset=["title", "text"])
df = df[df["text"].str.strip().str.len() > 0].reset_index(drop=True)
print("Cleaned shape:", df.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=df["label"].map({0: "Real", 1: "Fake"}), ax=ax[0], palette=["#2ecc71", "#e74c3c"])
ax[0].set_title("Class Balance")

df["word_count"] = df["text"].str.split().str.len()
sns.histplot(data=df, x="word_count", hue=df["label"].map({0: "Real", 1: "Fake"}),
             bins=60, ax=ax[1], binrange=(0, 2000))
ax[1].set_title("Article Length (words)")

plt.tight_layout()
plt.savefig("images/eda.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i, (lab, name) in enumerate([(0, "Real"), (1, "Fake")]):
    text = " ".join(df[df["label"] == lab]["text"].sample(2000, random_state=1))
    wc = WordCloud(width=700, height=400, background_color="white", stopwords=None).generate(text)
    ax[i].imshow(wc); ax[i].axis("off"); ax[i].set_title(f"{name} News WordCloud")
plt.tight_layout()
plt.savefig("images/wordcloud.png", dpi=150)
plt.show()

## 4. Text Preprocessing

⚠️ **Data leakage ka masla:** Is dataset mein real news aksar `WASHINGTON (Reuters) -` se shuru hoti hai.
Agar hum ye rakhein to model sirf "Reuters" dekh kar 99% accuracy le lega, jo asli kamyabi nahi.
Is liye hum ye publisher tags aur `subject` column hata rahe hain.

In [ ]:
def clean_text(text):
    text = str(text)
    # Leakage hatana: "CITY (Reuters) -" jaisa shuru ka hissa
    text = re.sub(r"^.{0,60}\(Reuters\)\s*-?\s*", "", text)
    text = re.sub(r"\bReuters\b", "", text, flags=re.I)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)      # links
    text = re.sub(r"[^a-z\s]", " ", text)             # punctuation, numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["content"] = df["title"].fillna("") + " " + df["text"]
df["clean"] = df["content"].apply(clean_text)
df[["clean", "label"]].head()

## 5. Train/Test split aur TF-IDF
80% training, 20% testing. TF-IDF sirf **train** par fit hoga (taake test ka data leak na ho).
Stopwords TF-IDF khud hata deta hai.

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    df["clean"], df["label"], df.index,
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"]
)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                        max_features=20000, min_df=5, max_df=0.9)
Xtr = tfidf.fit_transform(X_train)
Xte = tfidf.transform(X_test)
print("Train matrix:", Xtr.shape, "| Test matrix:", Xte.shape)

## 6. Models train karna

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=3),
    "Naive Bayes":         MultinomialNB(),
    "Linear SVM":          LinearSVC(C=1),
    "Random Forest":       RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
}

results, preds = [], {}
for name, model in models.items():
    print(f"Training {name} ...")
    model.fit(Xtr, y_train)
    p = model.predict(Xte)
    preds[name] = p
    results.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, p),
        "Precision": precision_score(y_test, p),
        "Recall":    recall_score(y_test, p),
        "F1-score":  f1_score(y_test, p),
    })

res_df = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
res_df.round(4)

## 7. Evaluation

In [ ]:
res_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0.8, 1.01))
plt.title("Model Comparison")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/model_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, p) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, p)
    ConfusionMatrixDisplay(cm, display_labels=["Real", "Fake"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.savefig("images/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

plt.figure(figsize=(7, 6))
for name, model in models.items():
    fpr, tpr, _ = roc_curve(y_test, get_scores(model, Xte))
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves"); plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/roc_curve.png", dpi=150)
plt.show()

In [ ]:
best_name = res_df.loc[0, "Model"]
print("Best model:", best_name, "\n")
print(classification_report(y_test, preds[best_name], target_names=["Real", "Fake"]))

## 8. Error Analysis 🔍
Model kahan galat hai aur kyun? Ye section recruiters ko sab se zyada impress karta hai.

In [ ]:
# Logistic Regression ke sab se important words
lr = models["Logistic Regression"]
feat = np.array(tfidf.get_feature_names_out())
coefs = lr.coef_[0]
top_fake = feat[np.argsort(coefs)[-15:]][::-1]
top_real = feat[np.argsort(coefs)[:15]]
print("🔴 Fake news ki taraf dhakelne wale words:\n", ", ".join(top_fake))
print("\n🟢 Real news ki taraf dhakelne wale words:\n", ", ".join(top_real))

In [ ]:
test_df = df.loc[idx_test].copy()
test_df["pred"] = preds[best_name]
wrong = test_df[test_df["pred"] != test_df["label"]]
print(f"Total galat predictions: {len(wrong)} / {len(test_df)}")

for _, row in wrong.sample(min(5, len(wrong)), random_state=1).iterrows():
    actual = "Fake" if row["label"] == 1 else "Real"
    pred = "Fake" if row["pred"] == 1 else "Real"
    print(f"\nActual: {actual} | Predicted: {pred}")
    print("Title:", row["title"])
    print("Text :", row["text"][:300], "...")

**Apne observations yahan likhein** (README mein bhi daalna):
- Galat predictions zyada tar kin articles par hui? (chhoti text, opinion pieces, satire?)
- Kya model sirf style dekh raha hai ya waqai facts?
- Limitation: ye model sirf is dataset (US politics, 2016-2017) par train hai, dusre topics par kamzor ho sakta hai.

## 9. Predict function

In [ ]:
final_model = models["Logistic Regression"]   # probability deta hai, demo ke liye behtar

def predict_news(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    prob_fake = final_model.predict_proba(vec)[0, 1]
    label = "FAKE" if prob_fake >= 0.5 else "REAL"
    return label, float(prob_fake)

label, p = predict_news("Scientists confirm that the moon is made of cheese, government hiding the truth!")
print(label, f"(fake probability: {p:.2%})")

## 10. Gradio Demo 🎛️

In [ ]:
import gradio as gr

def gradio_predict(text):
    if not text or len(text.split()) < 10:
        return {"Please enter at least 10 words": 1.0}
    _, p = predict_news(text)
    return {"Fake": p, "Real": 1 - p}

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=8, label="News article text (English)"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="📰 Fake News Detector",
    description="Paste an English news article and the model will predict if it is Real or Fake.",
)
demo.launch(share=True)

## 11. Images download karna (GitHub ke liye)
Ye cell `images/` folder ko zip karke download karega. Isay GitHub repo mein upload karein.

In [ ]:
!zip -qr images.zip images
from google.colab import files
files.download("images.zip")

## ✅ Conclusion
- 4 models compare kiye; sab se behtar model ka F1-score upar table mein hai.
- Data leakage (Reuters tag) hata kar zyada honest result nikala.
- Limitations: sirf English, sirf ek dataset, aur model facts verify nahi karta, sirf writing patterns seekhta hai.

**Future work:** BERT / DistilBERT fine-tuning, Urdu news dataset, cross-dataset testing.